# Ecoli Protein Localisation Classification

Where does an E. coli protein end up inside the cell? Seven sequence-derived scores predict its localisation site among eight sites. The catch is the same as in the yeast project: the classes are wildly uneven — cytoplasm has 143 proteins, while two sites have only 2 each.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Ecoli (id 39)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/39/data.csv')))
print(df.shape)
print('Proteins per localisation site:')
print(df['class'].value_counts())
df.head(3)


(336, 9)
Proteins per localisation site:
class
cp     143
im      77
pp      52
imU     35
om      20
omL      5
imS      2
imL      2
Name: count, dtype: int64


,Sequence,mcg,gvh,lip,chg,aac,alm1,alm2,class
0,AAT_ECOLI,0.49,0.29,0.48,0.5,0.56,0.24,0.35,cp
1,ACEA_ECOLI,0.07,0.40,0.48,0.5,0.54,0.35,0.44,cp
2,ACEK_ECOLI,0.56,0.40,0.48,0.5,0.49,0.37,0.46,cp


In [2]:

feats = ['mcg', 'gvh', 'lip', 'chg', 'aac', 'alm1', 'alm2']  # Sequence is just an ID and is dropped
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.8690
Logistic Regression accuracy: 0.8810
              precision    recall  f1-score   support

          cp       0.97      1.00      0.99        36
          im       0.74      0.74      0.74        19
         imS       0.00      0.00      0.00         1
         imU       0.58      0.78      0.67         9
          om       1.00      1.00      1.00         5
         omL       1.00      1.00      1.00         1
          pp       1.00      0.77      0.87        13

    accuracy                           0.87        84
   macro avg       0.76      0.75      0.75        84
weighted avg       0.87      0.87      0.87        84



/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which sequence scores localise a protein?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


alm1    0.260868
mcg     0.224600
gvh     0.181101
alm2    0.178552
aac     0.131106
lip     0.020927
chg     0.002847
dtype: float64


Winner: Logistic Regression
